# 07 — Player Role Fit & Tactical Match Analysis

This notebook connects **player value profiles** with **team tactical styles** to create a first-pass recruitment-oriented fit layer.

## Objectives

1. Load:
   - player value outputs from Notebook 05
   - team tactical outputs from Notebook 06
2. Build comparable feature spaces for:
   - player style / value profiles
   - team tactical demands
3. Estimate **player-to-team fit scores** using profile distance and weighted tactical alignment.
4. Produce:
   - best-fit team suggestions for each player
   - best-fit player suggestions for each team
   - role-oriented shortlist views
5. Persist the final fit tables for downstream reporting.

## Outputs

- `data/processed/player_team_fit.parquet`
- `data/processed/team_role_targets.parquet`
- `data/processed/player_role_profiles.parquet`
- `db/football_value.duckdb`
  - `player_team_fit`
  - `team_role_targets`
  - `player_role_profiles`


## 0. Setup

In [ ]:
from __future__ import annotations

import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
warnings.filterwarnings("ignore")


In [ ]:
try:
    import duckdb
except Exception as e:
    raise ImportError("duckdb is required. Install it with: pip install duckdb") from e

try:
    from sklearn.preprocessing import StandardScaler
    from sklearn.metrics.pairwise import cosine_similarity
    SKLEARN_AVAILABLE = True
except Exception:
    SKLEARN_AVAILABLE = False


## 1. Project paths

In [ ]:
def find_repo_root(start: Optional[Path] = None) -> Path:
    start = start or Path.cwd()
    for p in [start, *start.parents]:
        if (p / "README.md").exists():
            return p
    return start

REPO_ROOT = find_repo_root()
DATA_DIR = REPO_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
DB_DIR = REPO_ROOT / "db"
REPORTS_DIR = REPO_ROOT / "reports"

for d in [PROCESSED_DIR, DB_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DB_PATH = DB_DIR / "football_value.duckdb"

PLAYER_VALUE_PATH = PROCESSED_DIR / "player_value_table.parquet"
PLAYER_SEGMENTS_PATH = PROCESSED_DIR / "player_segments.parquet"
TEAM_TACTICAL_PATH = PROCESSED_DIR / "team_tactical_table.parquet"
TEAM_STYLE_CLUSTERS_PATH = PROCESSED_DIR / "team_style_clusters.parquet"

PLAYER_TEAM_FIT_PATH = PROCESSED_DIR / "player_team_fit.parquet"
TEAM_ROLE_TARGETS_PATH = PROCESSED_DIR / "team_role_targets.parquet"
PLAYER_ROLE_PROFILES_PATH = PROCESSED_DIR / "player_role_profiles.parquet"

REPO_ROOT, DB_PATH


## 2. Configuration

In [ ]:
@dataclass(frozen=True)
class Config:
    min_player_actions: int = 150
    exclude_same_team_matches: bool = True
    top_k_team_fits_per_player: int = 5
    top_k_player_fits_per_team: int = 10
    save_outputs: bool = True

CFG = Config()
CFG


## 3. Load required datasets

Expected inputs:
- `player_value_table.parquet`
- `player_segments.parquet`
- `team_tactical_table.parquet`
- `team_style_clusters.parquet`


In [ ]:
required_paths = [
    PLAYER_VALUE_PATH,
    PLAYER_SEGMENTS_PATH,
    TEAM_TACTICAL_PATH,
    TEAM_STYLE_CLUSTERS_PATH,
]

missing_paths = [str(p) for p in required_paths if not p.exists()]
if missing_paths:
    raise FileNotFoundError(
        "Missing required inputs. Run Notebooks 05 and 06 first:\n" + "\n".join(missing_paths)
    )

player_value = pd.read_parquet(PLAYER_VALUE_PATH)
player_segments = pd.read_parquet(PLAYER_SEGMENTS_PATH)
team_tactical = pd.read_parquet(TEAM_TACTICAL_PATH)
team_style_clusters = pd.read_parquet(TEAM_STYLE_CLUSTERS_PATH)

print("player_value:", player_value.shape)
print("player_segments:", player_segments.shape)
print("team_tactical:", team_tactical.shape)
print("team_style_clusters:", team_style_clusters.shape)


## 4. Build player role profile table

In [ ]:
player_role_profiles = player_value.copy()
player_role_profiles = player_role_profiles[
    player_role_profiles["n_actions"] >= CFG.min_player_actions
].copy()

role_feature_cols = [
    "actions_per_90",
    "progressive_actions_per_90",
    "xt_per_90",
    "vaep_per_90",
    "xt_per_100_actions",
    "vaep_per_100_actions",
    "pass_share",
    "carry_share",
    "dribble_share",
    "shot_share",
    "progression_score",
    "overall_value_score",
    "efficiency_score",
    "scouting_score",
]

base_player_cols = [
    "player",
    "primary_team",
    "n_actions",
    "n_matches",
    "rule_based_segment",
    "cluster_id",
]

player_role_profiles = player_role_profiles[base_player_cols + role_feature_cols].copy()
player_role_profiles.head()


## 5. Build team tactical demand table

In [ ]:
team_role_targets = team_tactical.copy()

team_role_targets["team_actions_per_90_target"] = team_role_targets["actions_per_90"]
team_role_targets["team_progressive_actions_per_90_target"] = team_role_targets["progressive_actions_per_90"]
team_role_targets["team_xt_per_90_target"] = team_role_targets["xt_per_90"]
team_role_targets["team_vaep_per_90_target"] = team_role_targets["vaep_per_90"]
team_role_targets["team_xt_per_100_actions_target"] = team_role_targets["xt_per_100_actions"]
team_role_targets["team_vaep_per_100_actions_target"] = team_role_targets["vaep_per_100_actions"]

team_role_targets["team_pass_share_target"] = team_role_targets["pass_share"]
team_role_targets["team_carry_share_target"] = team_role_targets["carry_share"]
team_role_targets["team_dribble_share_target"] = team_role_targets["dribble_share"]
team_role_targets["team_shot_share_target"] = team_role_targets["shot_share"]

team_role_targets["team_progression_score_target"] = team_role_targets["progression_style_score"]
team_role_targets["team_overall_value_score_target"] = (
    0.50 * team_role_targets["vaep_per_90_pct"] +
    0.50 * team_role_targets["xt_per_90_pct"]
)
team_role_targets["team_efficiency_score_target"] = (
    0.50 * team_role_targets["final_third_value_score"] +
    0.50 * team_role_targets["vaep_per_100_actions_pct"]
)

team_target_feature_cols = [
    "team_actions_per_90_target",
    "team_progressive_actions_per_90_target",
    "team_xt_per_90_target",
    "team_vaep_per_90_target",
    "team_xt_per_100_actions_target",
    "team_vaep_per_100_actions_target",
    "team_pass_share_target",
    "team_carry_share_target",
    "team_dribble_share_target",
    "team_shot_share_target",
    "team_progression_score_target",
    "team_overall_value_score_target",
    "team_efficiency_score_target",
]

team_role_targets[[
    "team",
    "rule_based_style",
    "cluster_id",
] + team_target_feature_cols].head()


## 6. Align feature spaces

In [ ]:
player_feature_matrix = player_role_profiles[role_feature_cols].copy()

team_feature_matrix = pd.DataFrame({
    "actions_per_90": team_role_targets["team_actions_per_90_target"],
    "progressive_actions_per_90": team_role_targets["team_progressive_actions_per_90_target"],
    "xt_per_90": team_role_targets["team_xt_per_90_target"],
    "vaep_per_90": team_role_targets["team_vaep_per_90_target"],
    "xt_per_100_actions": team_role_targets["team_xt_per_100_actions_target"],
    "vaep_per_100_actions": team_role_targets["team_vaep_per_100_actions_target"],
    "pass_share": team_role_targets["team_pass_share_target"],
    "carry_share": team_role_targets["team_carry_share_target"],
    "dribble_share": team_role_targets["team_dribble_share_target"],
    "shot_share": team_role_targets["team_shot_share_target"],
    "progression_score": team_role_targets["team_progression_score_target"],
    "overall_value_score": team_role_targets["team_overall_value_score_target"],
    "efficiency_score": team_role_targets["team_efficiency_score_target"],
    "scouting_score": team_role_targets["team_overall_value_score_target"],
})

player_feature_matrix.head(), team_feature_matrix.head()


## 7. Compute player-to-team fit scores

In [ ]:
if not SKLEARN_AVAILABLE:
    raise ImportError("scikit-learn is required for fit computation. Install it with: pip install scikit-learn")

scaler = StandardScaler()

combined = pd.concat([
    player_feature_matrix.assign(_kind="player"),
    team_feature_matrix.assign(_kind="team"),
], ignore_index=True)

combined_scaled = scaler.fit_transform(combined.drop(columns="_kind"))

player_scaled = combined_scaled[:len(player_feature_matrix)]
team_scaled = combined_scaled[len(player_feature_matrix):]

cos_sim = cosine_similarity(player_scaled, team_scaled)

dist_matrix = np.sqrt(((player_scaled[:, None, :] - team_scaled[None, :, :]) ** 2).sum(axis=2))
dist_score = 1 / (1 + dist_matrix)

final_fit_score = 0.60 * cos_sim + 0.40 * dist_score

final_fit_score.shape


## 8. Build player-team fit table

In [ ]:
fit_rows = []

player_reset = player_role_profiles.reset_index(drop=True)
team_reset = team_role_targets.reset_index(drop=True)

for i, player_row in player_reset.iterrows():
    for j, team_row in team_reset.iterrows():
        fit_rows.append({
            "player": player_row["player"],
            "player_current_team": player_row["primary_team"],
            "player_segment": player_row["rule_based_segment"],
            "team": team_row["team"],
            "team_style": team_row["rule_based_style"],
            "team_cluster_id": team_row["cluster_id"],
            "cosine_fit": float(cos_sim[i, j]),
            "distance_fit": float(dist_score[i, j]),
            "fit_score": float(final_fit_score[i, j]),
        })

player_team_fit = pd.DataFrame(fit_rows)

if CFG.exclude_same_team_matches:
    player_team_fit = player_team_fit[
        player_team_fit["player_current_team"] != player_team_fit["team"]
    ].copy()

player_team_fit = player_team_fit.sort_values(["player", "fit_score"], ascending=[True, False]).reset_index(drop=True)
player_team_fit.head(20)


In [ ]:
# Scale fit score to [0,1] for interpretability
player_team_fit["fit_score_scaled"] = (
    player_team_fit["fit_score"] - player_team_fit["fit_score"].min()
) / (
    player_team_fit["fit_score"].max() - player_team_fit["fit_score"].min()
)

player_team_fit.head()

## 9. Rank best-fit teams for each player

In [ ]:
# Rank best-fit teams for each player
player_team_fit["team_rank_for_player"] = (
    player_team_fit.groupby("player")["fit_score_scaled"]
    .rank(method="first", ascending=False)
)

best_fit_teams_per_player = player_team_fit[
    player_team_fit["team_rank_for_player"] <= CFG.top_k_team_fits_per_player
].copy()

best_fit_teams_per_player = best_fit_teams_per_player.sort_values(
    ["player", "team_rank_for_player", "fit_score_scaled"],
    ascending=[True, True, False]
).reset_index(drop=True)

best_fit_teams_per_player.head(20)

## 10. Rank best-fit players for each team

In [ ]:
# Rank best-fit players for each team
player_team_fit["player_rank_for_team"] = (
    player_team_fit.groupby("team")["fit_score_scaled"]
    .rank(method="first", ascending=False)
)

best_fit_players_per_team = player_team_fit[
    player_team_fit["player_rank_for_team"] <= CFG.top_k_player_fits_per_team
].copy()

best_fit_players_per_team = best_fit_players_per_team.sort_values(
    ["team", "player_rank_for_team", "fit_score_scaled"],
    ascending=[True, True, False]
).reset_index(drop=True)

best_fit_players_per_team.head(20)

## 11. Team-role shortlist view

In [ ]:
team_role_shortlist = (
    best_fit_players_per_team
    .merge(
        player_value[[
            "player",
            "primary_team",
            "xt_per_90",
            "vaep_per_90",
            "scouting_score",
            "progression_score",
            "efficiency_score",
        ]],
        on="player",
        how="left"
    )
    .sort_values(["team", "fit_score"], ascending=[True, False])
    .reset_index(drop=True)
)

team_role_shortlist.head(20)


## 12. Example team-target view

In [ ]:
example_team_targets = team_role_shortlist[[
    "team",
    "player",
    "player_current_team",
    "player_segment",
    "fit_score",
    "scouting_score",
    "xt_per_90",
    "vaep_per_90",
]].copy()

example_team_targets.head(20)


## 13. Visualise fit map for top-value players

In [ ]:
top_players = (
    player_value.sort_values("scouting_score", ascending=False)["player"]
    .head(12)
    .tolist()
)

fit_subset = player_team_fit[player_team_fit["player"].isin(top_players)].copy()

fig, ax = plt.subplots(figsize=(11, 7))
for player in top_players:
    sub = fit_subset[fit_subset["player"] == player].nlargest(1, "fit_score")
    if not sub.empty:
        r = sub.iloc[0]
        ax.scatter(r["fit_score"], r["cosine_fit"], s=60)
        ax.annotate(f'{r["player"]} → {r["team"]}', (r["fit_score"], r["cosine_fit"]), fontsize=8)

ax.set_title("Top Player Best-Fit Matches")
ax.set_xlabel("Final fit score")
ax.set_ylabel("Cosine fit")
plt.show()


## 14. Example team radar comparison

In [ ]:
def radar_plot(team_name, shortlist_df, role_df, team_df):
    subset = shortlist_df[shortlist_df["team"] == team_name].head(3)
    if subset.empty:
        print("No shortlist data for this team.")
        return

    metrics = [
        "actions_per_90",
        "progressive_actions_per_90",
        "xt_per_90",
        "vaep_per_90",
        "progression_score",
    ]

    team_row = team_df[team_df["team"] == team_name]
    if team_row.empty:
        print("Team not found.")
        return

    angles = np.linspace(0, 2 * np.pi, len(metrics), endpoint=False).tolist()
    angles += angles[:1]

    fig = plt.figure(figsize=(8, 8))
    ax = plt.subplot(111, polar=True)

    team_values = [
        float(team_row["actions_per_90"].iloc[0]),
        float(team_row["progressive_actions_per_90"].iloc[0]),
        float(team_row["xt_per_90"].iloc[0]),
        float(team_row["vaep_per_90"].iloc[0]),
        float(team_row["progression_style_score"].iloc[0]),
    ]
    team_values = np.array(team_values, dtype=float)
    team_values = team_values / np.maximum(team_values.max(), 1e-6)
    team_values = team_values.tolist() + [team_values.tolist()[0]]

    ax.plot(angles, team_values, linewidth=2, label=f"{team_name} target")
    ax.fill(angles, team_values, alpha=0.08)

    for _, cand in subset.iterrows():
        prow = role_df[role_df["player"] == cand["player"]]
        if prow.empty:
            continue
        values = [
            float(prow["actions_per_90"].iloc[0]),
            float(prow["progressive_actions_per_90"].iloc[0]),
            float(prow["xt_per_90"].iloc[0]),
            float(prow["vaep_per_90"].iloc[0]),
            float(prow["progression_score"].iloc[0]),
        ]
        values = np.array(values, dtype=float)
        values = values / np.maximum(values.max(), 1e-6)
        values = values.tolist() + [values.tolist()[0]]

        ax.plot(angles, values, linewidth=1.8, label=cand["player"])

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(metrics)
    ax.set_title(f"Team Fit Radar — {team_name}")
    ax.legend(loc="upper right", bbox_to_anchor=(1.45, 1.10))
    plt.show()

example_team = team_role_shortlist["team"].value_counts().index[0]
radar_plot(example_team, team_role_shortlist, player_role_profiles, team_tactical)


## 15. Top recruitment fit tables

In [ ]:
top_player_fit_table = best_fit_teams_per_player.merge(
    player_value[["player", "scouting_score", "xt_per_90", "vaep_per_90"]],
    on="player",
    how="left"
).sort_values(["player", "fit_score"], ascending=[True, False])

top_player_fit_table.head(20)


In [ ]:
top_team_fit_table = team_role_shortlist.sort_values(
    ["team", "fit_score", "scouting_score"],
    ascending=[True, False, False]
)

top_team_fit_table.head(20)


## 16. Save outputs

In [ ]:
if CFG.save_outputs:
    player_team_fit.to_parquet(PLAYER_TEAM_FIT_PATH, index=False)
    team_role_targets.to_parquet(TEAM_ROLE_TARGETS_PATH, index=False)
    player_role_profiles.to_parquet(PLAYER_ROLE_PROFILES_PATH, index=False)

    try:
        con.close()
    except:
        pass

    con = duckdb.connect(str(DB_PATH))
    con.execute("CREATE OR REPLACE TABLE player_team_fit AS SELECT * FROM read_parquet(?)", [str(PLAYER_TEAM_FIT_PATH)])
    con.execute("CREATE OR REPLACE TABLE team_role_targets AS SELECT * FROM read_parquet(?)", [str(TEAM_ROLE_TARGETS_PATH)])
    con.execute("CREATE OR REPLACE TABLE player_role_profiles AS SELECT * FROM read_parquet(?)", [str(PLAYER_ROLE_PROFILES_PATH)])
    con.close()

    print(f"Saved player-team fit table to: {PLAYER_TEAM_FIT_PATH}")
    print(f"Saved team role targets to:     {TEAM_ROLE_TARGETS_PATH}")
    print(f"Saved player role profiles to:  {PLAYER_ROLE_PROFILES_PATH}")
    print(f"Updated DuckDB at:              {DB_PATH}")


## 17. Preview from DuckDB

In [ ]:
con = duckdb.connect(str(DB_PATH))

summary_sql = '''
SELECT
    COUNT(*) AS n_player_team_pairs,
    AVG(fit_score) AS avg_fit_score,
    MAX(fit_score) AS max_fit_score,
    MIN(fit_score) AS min_fit_score
FROM player_team_fit
'''
preview = con.execute(summary_sql).df()
con.close()

preview


# Conclusions — Player-Team Tactical Fit Analysis

## Overview

This notebook extends the analytical pipeline from **player value modelling** and **team tactical profiling** into a recruitment-oriented **player–team compatibility framework**.

Using the outputs of previous stages of the project:

* **player value metrics** (xT, VAEP, progression indicators)
* **player role segmentation**
* **team tactical style modelling**

we construct a **player-to-team fit model** that measures how well an individual player profile aligns with the tactical characteristics of a given team.

The resulting framework produces **16,700 player–team combinations**, enabling systematic exploration of tactical compatibility across the dataset.

---

# Player–Team Fit Modelling

Player compatibility with each team is estimated using a combination of two similarity metrics:

### Cosine similarity

Captures how similar the **player style vector** is to the **team tactical demand vector**.

This focuses on:

* progression behaviour
* action distribution
* offensive value creation

### Distance-based similarity

A Euclidean distance transformation provides a complementary measure that captures the **absolute proximity between player and team profiles**.

### Final fit score

Both metrics are combined into a single compatibility measure:

[
FitScore = 0.6 \times CosineSimilarity + 0.4 \times DistanceScore
]

To facilitate interpretation, the score is then rescaled:

[
FitScore_{scaled} \in [0,1]
]

where:

| Fit Score   | Interpretation           |
| ----------- | ------------------------ |
| 0.80 – 1.00 | Excellent tactical match |
| 0.60 – 0.80 | Strong fit               |
| 0.40 – 0.60 | Moderate fit             |
| < 0.40      | Weak tactical alignment  |

---

# Example: Málaga Tactical Fit

The highest compatibility results in the dataset illustrate how the model identifies players whose profiles match the tactical characteristics of specific teams.

For instance, Málaga appears as a **Direct progression side**, meaning that its style is characterised by:

* vertical ball progression
* fast territorial advancement
* value creation through forward movement

The model identifies several players with **very high compatibility scores** for this tactical profile:

| Player               | Fit Score (scaled) |
| -------------------- | ------------------ |
| Hugo Lloris          | 1.00               |
| Yuri Zhirkov         | 0.998              |
| William Troost-Ekong | 0.994              |
| Víctor Valdés        | 0.982              |
| Yann Sommer          | 0.978              |

These players share similar stylistic characteristics consistent with the tactical profile identified for Málaga.

This demonstrates how the model can surface **players whose action profiles align with a team's strategic approach to possession progression**.

---

# Tactical Radar Interpretation

The radar comparison between **team tactical targets** and **player profiles** provides an intuitive visual representation of compatibility.

The radar includes the following dimensions:

* actions per 90
* progressive actions per 90
* expected threat generation (xT)
* action value (VAEP)
* progression score

Players whose radar shapes closely resemble the team's tactical profile tend to obtain the highest fit scores.

This visual layer helps translate quantitative compatibility scores into **interpretable tactical insights**.

---

# Analytical Value

This modelling layer enables several recruitment-oriented analytical tasks:

### Tactical scouting

Identify players whose **style naturally fits a team's system**, even if they currently play in a different tactical environment.

### Player contextualisation

Distinguish between:

* players who are **globally strong performers**
* players who are **particularly suited to a specific tactical context**

### Recruitment shortlisting

Generate **candidate lists for each team** based on tactical compatibility rather than only raw performance metrics.

---

# Project Contribution

With this notebook, the project completes a full football analytics pipeline:

```
Event data
   ↓
xT possession value model
   ↓
VAEP action valuation
   ↓
Player value modelling
   ↓
Player role segmentation
   ↓
Team tactical profiling
   ↓
Player–team compatibility modelling
```

This architecture mirrors the analytical workflows used in professional football research environments, where recruitment decisions combine:

* **player quality**
* **role profile**
* **team tactical identity**

---

# Key Takeaway

The most valuable players are not always the best fits for every team.

This notebook demonstrates how **data-driven compatibility modelling** can identify players whose profiles align with a team’s tactical demands, providing a structured foundation for **modern recruitment analytics**.